# Banking AI Assistant

## Phase 3 — Document Preparation for RAG

This notebook prepares banking FAQ documents for:

- Semantic Search
- FAISS Vector Database
- Retrieval-Augmented Generation (RAG)
- Enterprise Banking Chatbot

Output:
- rag_documents.pkl
- rag_documents.csv

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

import pickle
import os

from langchain_core.documents import Document

from pathlib import Path

pd.set_option("display.max_colwidth", None)

### Load Processed Dataset

In [3]:
df = pd.read_csv(
    "../data/processed_data/02_banking_processed.csv"
)

print("Dataset Shape:", df.shape)

df.sample(1)

Dataset Shape: (2331, 9)


,Category,Question,Answer,question_length,answer_length,clean_question,standard_category,processed_question,label
596,insurance,Do I need to pay for hospitalisation,"If you are admitted in any of our network hospitals, you can avail a cashless facility. We will directly reimburse all the admissible expenses to the hospital. However, in case of non-network hospitals you will have to settle hospital bills at the time of discharge and consequently, the same will be reimbursed to you by us.",36,325,need pay hospitalisation,Investments & Insurance,do i need to pay for hospitalisation,3


### Dataset Overview

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2331 entries, 0 to 2330
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Category            2331 non-null   object
 1   Question            2331 non-null   object
 2   Answer              2331 non-null   object
 3   question_length     2331 non-null   int64 
 4   answer_length       2331 non-null   int64 
 5   clean_question      2331 non-null   object
 6   standard_category   2331 non-null   object
 7   processed_question  2331 non-null   object
 8   label               2331 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 164.0+ KB


In [5]:
df.columns

Index(['Category', 'Question', 'Answer', 'question_length', 'answer_length',
       'clean_question', 'standard_category', 'processed_question', 'label'],
      dtype='object')

### Check Missing Values

In [6]:
df.isnull().sum()

Category              0
Question              0
Answer                0
question_length       0
answer_length         0
clean_question        0
standard_category     0
processed_question    0
label                 0
dtype: int64

#### Remove Duplicate Question-Answer Pairs

In [7]:
before = len(df)

df = df.drop_duplicates(subset=["Question", "Answer"])

after = len(df)

print("Before:", before)
print("After :", after)
print("Removed:", before - after)

Before: 2331
After : 2331
Removed: 0


### Category Distribution

In [8]:
df["standard_category"].value_counts()

standard_category
Investments & Insurance    693
Customer Support           359
Retail Banking             356
Cards & Payments           324
Loans                      318
Digital & Security         281
Name: count, dtype: int64

### Create Metadata Fields

In [9]:
df["document_id"] = range(1, len(df) + 1)

### Create Retrieval Text

In [10]:
df["retrieval_text"] = (
    "Category: " + df["standard_category"] + 
    "\n\nQuestion: " + df["Question"] + 
    "\n\nAnswer: " + df["Answer"]
)

### Inspect Retrieval Text

In [11]:
print(df["retrieval_text"].iloc[0])

Category: Retail Banking

Question: What are the documents required for opening a Current Account of a sole proprietorship firm

Answer: Following documents are required to open a Current Account of a sole proprietorship entity: Proof of existence in the name of firm Proof of address in the name of firm KYC of the proprietor Any two of the below listed documents shall be obtained for establishing proof of existence. Registration certificate/license issued by Municipal authorities such as Shop & Establishment Certificate/Trade License CST/VAT/Service Tax Certificate or Letter Of Registration for CST/VAT/Service Tax Certificate/Registration document issued by Professional Tax authorities Valid Business License or Certificate Of Registration issued by State/Central Government authority (validity would include the grace period for renewal as mentioned in the certificate) RBI/SEBI Registration Certificate License issued by Food and Drug Control Authorities Import - Export certificate (IEC C

### Create LangChain Documents

In [12]:
documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=row["retrieval_text"],
        metadata={

            "document_id": int(row["document_id"]),
            "category": row["standard_category"],
            "original_category": row["Category"],
            "question": row["Question"]
        }
    )

    documents.append(doc)

### Verify Documents

In [13]:
len(documents)

2331

In [14]:
documents[0]

Document(metadata={'document_id': 1, 'category': 'Retail Banking', 'original_category': 'accounts', 'question': 'What are the documents required for opening a Current Account of a sole proprietorship firm'}, page_content='Category: Retail Banking\n\nQuestion: What are the documents required for opening a Current Account of a sole proprietorship firm\n\nAnswer: Following documents are required to open a Current Account of a sole proprietorship entity: Proof of existence in the name of firm Proof of address in the name of firm KYC of the proprietor Any two of the below listed documents shall be obtained for establishing proof of existence. Registration certificate/license issued by Municipal authorities such as Shop & Establishment Certificate/Trade License CST/VAT/Service Tax Certificate or Letter Of Registration for CST/VAT/Service Tax Certificate/Registration document issued by Professional Tax authorities Valid Business License or Certificate Of Registration issued by State/Central G

### Sample Document Viewer

In [15]:
for i in range(3):
    print("="*100)
    print(documents[i])
    print()

page_content='Category: Retail Banking

Question: What are the documents required for opening a Current Account of a sole proprietorship firm

Answer: Following documents are required to open a Current Account of a sole proprietorship entity: Proof of existence in the name of firm Proof of address in the name of firm KYC of the proprietor Any two of the below listed documents shall be obtained for establishing proof of existence. Registration certificate/license issued by Municipal authorities such as Shop & Establishment Certificate/Trade License CST/VAT/Service Tax Certificate or Letter Of Registration for CST/VAT/Service Tax Certificate/Registration document issued by Professional Tax authorities Valid Business License or Certificate Of Registration issued by State/Central Government authority (validity would include the grace period for renewal as mentioned in the certificate) RBI/SEBI Registration Certificate License issued by Food and Drug Control Authorities Import - Export cert

### Create RAG CSV Dataset

In [18]:
rag_df = df[[
    "document_id",
    "standard_category",
    "Question",
    "Answer",
    "retrieval_text"
]]

rag_df.sample(2)

,document_id,standard_category,Question,Answer,retrieval_text
335,336,Cards & Payments,What currencies are available,There are ten currencies offered: Australian Dollar (AUD) Dirhams (AED) Canadian Dollar (CAD) Euro (EUR) Japanese Yen (JPY) Singapore Dollars (SGD) Sterling Pound (GBP) Swedish Krona (SEK) Swiss Franc (CHF) US Dollar (USD) View more,Category: Cards & Payments\n\nQuestion: What currencies are available\n\nAnswer: There are ten currencies offered: Australian Dollar (AUD) Dirhams (AED) Canadian Dollar (CAD) Euro (EUR) Japanese Yen (JPY) Singapore Dollars (SGD) Sterling Pound (GBP) Swedish Krona (SEK) Swiss Franc (CHF) US Dollar (USD) View more
289,290,Cards & Payments,Are there charges I should know about M&N Bank Rewards Debit Card,Another benefit of the Rewards Debit card is that there are absolutely no transaction charges when shopping at a merchant location.,Category: Cards & Payments\n\nQuestion: Are there charges I should know about M&N Bank Rewards Debit Card\n\nAnswer: Another benefit of the Rewards Debit card is that there are absolutely no transaction charges when shopping at a merchant location.


### Save RAG CSV

In [19]:
output_path = ("../data/processed_data/03_rag_documents.csv")
rag_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: ../data/processed_data/03_rag_documents.csv


### Save LangChain Documents

In [20]:
output_path = ("../data/processed_data/03_rag_documents.pkl")

with open(output_path, "wb") as f:
    pickle.dump(documents, f)

print("Saved:", output_path)

Saved: ../data/processed_data/03_rag_documents.pkl


### Reload Verification

In [21]:
with open("../data/processed_data/03_rag_documents.pkl", "rb") as f:
    loaded_documents = pickle.load(f)

print("Documents Loaded:", len(loaded_documents))

Documents Loaded: 2331


### Statistics for RAG

In [22]:
rag_stats = pd.DataFrame({
    "Metric":[
        "Total Documents",
        "Unique Categories",
        "Average Question Length",
        "Average Answer Length"
    ],

    "Value":[
        len(df),
        df["standard_category"].nunique(),
        round(df["question_length"].mean(), 2),
        round(df["answer_length"].mean(), 2
        )
    ]
})

rag_stats

,Metric,Value
0,Total Documents,2331.00
1,Unique Categories,6.00
2,Average Question Length,46.81
3,Average Answer Length,234.48


### Export Metadata Report

In [24]:
metadata_report = (
    df.groupby("standard_category")
    .agg(
        total_documents=("document_id", "count"),
        avg_question_length=("question_length", "mean"),
        avg_answer_length=("answer_length", "mean")
    )
    .reset_index()
)
metadata_report

,standard_category,total_documents,avg_question_length,avg_answer_length
0,Cards & Payments,324,58.787037,283.959877
1,Customer Support,359,32.674095,151.373259
2,Digital & Security,281,38.195730,166.868327
3,Investments & Insurance,693,48.600289,257.523810
4,Loans,318,47.534591,225.603774
5,Retail Banking,356,52.814607,289.716292


### Save Metadata Report

In [25]:
metadata_report.to_csv("../data/processed_data/03_rag_metadata_report.csv", index=False)

print("Metadata Report Saved Successfully")

Metadata Report Saved Successfully


## Key Insights

### Dataset Prepared for Enterprise RAG

- Removed duplicate Question-Answer pairs.
- Created unified retrieval text using:
  - Category
  - Question
  - Answer

### Metadata Added

Each document contains:

- document_id
- standard_category
- original_category
- question

### LangChain Document Objects Created

These documents are now ready for:

1. Semantic Embedding Generation
2. FAISS Vector Database Creation
3. Retrieval-Augmented Generation (RAG)
4. Groundedness Validation
5. Hallucination Detection

### Output Files Generated

- 03_rag_documents.csv
- 03_rag_documents.pkl
- 03_rag_metadata_report.csv

This dataset will serve as the knowledge base for the Banking AI Assistant.